# ADC Toxicity Classifier
Full pipeline: data loading, LOAOCV evaluation, final model training, SHAP analysis.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, roc_auc_score, f1_score
from src.data_loader import load_data
from src.preprocessor import build_preprocessor, FEATURE_COLS, TARGET_REG, TARGET_CLF
from src.train import run_loaocv, RF_REG_PARAMS, RF_CLF_PARAMS, XGB_REG_PARAMS, XGB_CLF_PARAMS

df = load_data('data/TADC_Complete_v3_avec_S.xlsx')
print(df.shape)
df[['P','D','H','B','L','E','V','S(payload,organe)','%G\u22653 observ\u00e9','Y binaire (G\u22653 >10%)']].describe()

In [ ]:
print('Class distribution (Y binaire):')
print(df['Y binaire (G\u22653 >10%)'].value_counts())
print('\nADC row counts:')
print(df['ADC'].value_counts())

In [ ]:
reg_results = run_loaocv(df, task='regression')
print(f"Regression LOAOCV \u2014 MAE: {reg_results['mae']:.2f}  RMSE: {reg_results['rmse']:.2f}  R\u00b2: {reg_results['r2']:.3f}")

In [ ]:
clf_results = run_loaocv(df, task='classification')
print(f"Classification LOAOCV \u2014 AUC: {clf_results['auc']:.3f}  F1: {clf_results['f1']:.3f}  Acc: {clf_results['accuracy']:.3f}")

In [ ]:
baseline_pred_reg = df['T-ADC v3 = \u03a3\u00d7V\u00d7S'].values
baseline_true_reg = df[TARGET_REG].values
baseline_mae = mean_absolute_error(baseline_true_reg, baseline_pred_reg)
print(f'Baseline (T-ADC v3) MAE: {baseline_mae:.2f}')
print(f'ML LOAOCV MAE:          {reg_results["mae"]:.2f}')

In [ ]:
preprocessor = build_preprocessor()
X_all = preprocessor.fit_transform(df[FEATURE_COLS])
y_reg_all = df[TARGET_REG].values.astype(float)
y_clf_all = df[TARGET_CLF].values.astype(int)

# Regression
gs_reg = GridSearchCV(
    RandomForestRegressor(random_state=42),
    RF_REG_PARAMS, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1
)
gs_reg.fit(X_all, y_reg_all)
best_reg = gs_reg.best_estimator_

# Classification
gs_clf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    RF_CLF_PARAMS, cv=5, scoring='roc_auc', n_jobs=-1
)
gs_clf.fit(X_all, y_clf_all)
best_clf = gs_clf.best_estimator_

joblib.dump(best_reg, 'models/model_regression.pkl')
joblib.dump(best_clf, 'models/model_classification.pkl')
joblib.dump(preprocessor, 'models/preprocessor.pkl')
print('Models saved.')

In [ ]:
explainer_reg = shap.TreeExplainer(best_reg)
shap_values_reg = explainer_reg.shap_values(X_all)

feature_names = (
    list(preprocessor.named_transformers_['num'].get_feature_names_out())
    + list(preprocessor.named_transformers_['cat'].get_feature_names_out())
)

shap.summary_plot(shap_values_reg, X_all, feature_names=feature_names, show=True)

In [ ]:
explainer_clf = shap.TreeExplainer(best_clf)
shap_values_clf = explainer_clf.shap_values(X_all)
if isinstance(shap_values_clf, list):
    shap_values_clf = shap_values_clf[1]  # positive class

shap.summary_plot(shap_values_clf, X_all, feature_names=feature_names, show=True)